In [6]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)

In [7]:
DB_HOST = os.getenv("DB_HOST", "db")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "weather_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [8]:
engine = create_engine(DATABASE_URL)

In [9]:
with engine.connect() as conn:
    loc_df = pd.read_sql("select * from location", conn)
    weather_df = pd.read_sql("select * from weather_observation", conn)

In [10]:
weather_df.head()

,id,location_id,observed_at,temperature,feels_like,humidity,pressure,wind_speed,weather_description,weather_code,raw_json,created_at
0,7cc67643-b3cc-4dba-9a33-d7d3f86bd1b5,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 04:00:00+00:00,4.2,2.0,93.0,1001.0,4.2,None,3,None,2026-05-15 17:26:11.758867+00:00
1,ed70cecd-72f7-4fcc-a9f1-18e0533a5b66,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 05:00:00+00:00,4.2,1.9,93.0,1001.5,5.4,None,3,None,2026-05-15 17:26:11.758867+00:00
2,4b178fa5-df7b-481b-9a03-1242275c67d5,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 06:00:00+00:00,4.2,1.8,96.0,1001.7,6.5,None,51,None,2026-05-15 17:26:11.758867+00:00
3,d7977da2-6fb4-4df1-ad28-b86d57054e1a,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 07:00:00+00:00,4.4,2.2,96.0,1001.7,5.2,None,55,None,2026-05-15 17:26:11.758867+00:00
4,318e0712-f3e4-4964-89bc-da25576eed64,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 08:00:00+00:00,4.4,2.0,95.0,1001.8,6.4,None,51,None,2026-05-15 17:26:11.758867+00:00


In [12]:
df = loc_df.merge(weather_df, left_on='id', right_on='location_id', suffixes=['_loc', '_wea'])

In [13]:
seattle = df[df['city'] == 'Seattle']

In [14]:
seattle.shape

(2880, 19)

In [15]:
seattle.head()

,id_loc,city,state,country,latitude,longitude,created_at_loc,id_wea,location_id,observed_at,temperature,feels_like,humidity,pressure,wind_speed,weather_description,weather_code,raw_json,created_at_wea
0,20ac3297-5f99-4dc0-8383-0ad590b39c08,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 17:26:10.280675+00:00,7cc67643-b3cc-4dba-9a33-d7d3f86bd1b5,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 04:00:00+00:00,4.2,2.0,93.0,1001.0,4.2,None,3,None,2026-05-15 17:26:11.758867+00:00
1,20ac3297-5f99-4dc0-8383-0ad590b39c08,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 17:26:10.280675+00:00,ed70cecd-72f7-4fcc-a9f1-18e0533a5b66,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 05:00:00+00:00,4.2,1.9,93.0,1001.5,5.4,None,3,None,2026-05-15 17:26:11.758867+00:00
2,20ac3297-5f99-4dc0-8383-0ad590b39c08,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 17:26:10.280675+00:00,4b178fa5-df7b-481b-9a03-1242275c67d5,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 06:00:00+00:00,4.2,1.8,96.0,1001.7,6.5,None,51,None,2026-05-15 17:26:11.758867+00:00
3,20ac3297-5f99-4dc0-8383-0ad590b39c08,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 17:26:10.280675+00:00,d7977da2-6fb4-4df1-ad28-b86d57054e1a,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 07:00:00+00:00,4.4,2.2,96.0,1001.7,5.2,None,55,None,2026-05-15 17:26:11.758867+00:00
4,20ac3297-5f99-4dc0-8383-0ad590b39c08,Seattle,WA,USA,47.6062,-122.3321,2026-05-15 17:26:10.280675+00:00,318e0712-f3e4-4964-89bc-da25576eed64,20ac3297-5f99-4dc0-8383-0ad590b39c08,2026-01-02 08:00:00+00:00,4.4,2.0,95.0,1001.8,6.4,None,51,None,2026-05-15 17:26:11.758867+00:00


In [19]:
seattle['observed_date'] = seattle['observed_at'].dt.strftime('%Y/%m/%d')

In [21]:
seattle.groupby("observed_date")[['temperature', 'feels_like']].mean()

,temperature,feels_like
observed_date,,
2026/01/01,2.495833,0.233333
2026/01/02,5.095833,3.208333
2026/01/03,8.145833,6.320833
2026/01/04,8.191667,5.745833
2026/01/05,5.829167,3.487500
...,...,...
2026/04/26,11.537500,9.012500
2026/04/27,11.679167,9.212500
2026/04/28,10.283333,8.095833
